In [1]:
import numpy as np
import pandas as pd
pd.options.display.float_format = '{:.2f}'.format
import warnings
warnings.filterwarnings("ignore") # specify to ignore warning messages

In [2]:
def forecast_accuracy(forecast, actual):
    """
    Calculates accuracy by calculating mean absolute percentage error, mean absolute error, 
    and root-mean-squared error.
    
    forecast: forecast dataframe containing the results from a model.
    actual  : test set of a particular patient, with same dates as the forecast dataframe.
    
    output  : mape: mean absolute percentage error
              mae : mean absolute error
              rsme: root mean square error
    """
    forecast = forecast.values
    mape = np.mean(np.abs(forecast - actual)/np.abs(actual))  # MAPE
    mae = np.mean(np.abs(forecast - actual))    # MAE
    rmse = np.mean((forecast - actual)**2)**.5  # RMSE
    return mape, mae, rmse

In [3]:
df_vitals_combined_SBP_predicted = pd.read_csv('df_vitals_combined_SBP_predicted.csv')

df_vitals_combined_HR_predicted = pd.read_csv('df_vitals_combined_HR_predicted.csv')

In [4]:
vitals_combined_interpolated = pd.read_csv('vitals_combined_interpolated.csv')

# Drops batches containing less than 30 observations.
df_recordings_per_patient = vitals_combined_interpolated.groupby(['patient_uuid', 'batch']).count().reset_index()
low_batches = df_recordings_per_patient[(df_recordings_per_patient.HR <= 30)][['patient_uuid','batch']]

for i in range(len(low_batches)):
    low_patient = low_batches.patient_uuid.iloc[i]
    low_batch = low_batches.batch.iloc[i]
    vitals_combined_interpolated.drop(vitals_combined_interpolated[(vitals_combined_interpolated.patient_uuid == low_patient) & (vitals_combined_interpolated.batch == low_batch)].index, inplace=True)
    
idx = vitals_combined_interpolated.groupby(['patient_uuid'])['batch'].transform(max) == vitals_combined_interpolated['batch']

vitals_combined_interpolated = vitals_combined_interpolated[idx]

df_vitals_combined_SBP_test = vitals_combined_interpolated[['patient_uuid','reading_date','SBP']]

df_vitals_combined_HR_test = vitals_combined_interpolated[['patient_uuid','reading_date','HR']]

In [5]:
def get_metrics(df_predicted, df_test, pred_variable: str):
    if not pred_variable in ('SBP', 'HR'):
        throw('either SBP or HR must be the prediction variable')
        
    df_vitals_metrics =  pd.DataFrame(columns = ['patient_uuid','mape_arima','mae_arima','rmse_arima'])
    for patient_uuid in df_predicted['patient_uuid'].unique():
        df_predicted_patient = df_predicted[df_predicted['patient_uuid'] == patient_uuid]
        df_predicted_patient = df_predicted_patient[['reading_date',pred_variable]]
        df_predicted_patient['reading_date'] = pd.to_datetime(df_predicted_patient['reading_date'])
        df_predicted_patient.sort_values(by='reading_date',inplace=True)
        df_predicted_patient = df_predicted_patient.tail(7)
        df_predicted_patient.set_index('reading_date',inplace=True)

        df_test_patient = df_test[df_test['patient_uuid'] == patient_uuid]
        df_test_patient = df_test_patient[['reading_date',pred_variable]]
        df_test_patient['reading_date'] = pd.to_datetime(df_test_patient['reading_date'])
        df_test_patient.sort_values(by='reading_date',inplace=True)
        df_test_patient = df_test_patient.tail(7)
        df_test_patient.set_index('reading_date',inplace=True)

        mape, mae, rmse = forecast_accuracy(df_predicted_patient, df_test_patient)
        mape = float(str(mape.get_values()).strip('[]'))
        mae = float(str(mae.get_values()).strip('[]'))
        rmse = float(str(rmse.get_values()).strip('[]'))
        df_vitals_metrics.loc[len(df_vitals_metrics)] = [patient_uuid,mape,mae,rmse]
        
    return df_vitals_metrics


In [6]:
df_vitals_metrics_SBP = get_metrics(df_vitals_combined_SBP_predicted, df_vitals_combined_SBP_test, pred_variable = 'SBP')

df_vitals_metrics_HR = get_metrics(df_vitals_combined_HR_predicted, df_vitals_combined_HR_test, pred_variable = 'HR')

In [7]:
df_vitals_metrics_arima = pd.merge(df_vitals_metrics_SBP,df_vitals_metrics_HR,on=['patient_uuid'],how='inner')

df_vitals_metrics_arima.columns = ['patient_uuid','mape_arima_SBP','mae_arima_SBP','rmse_arima_SBP','mape_arima_HR','mae_arima_HR','rmse_arima_HR']

In [8]:
df_vitals_metrics_arima[~((df_vitals_metrics_arima['mape_arima_SBP']>1)|(df_vitals_metrics_arima['mape_arima_HR']>1))]

,patient_uuid,mape_arima_SBP,mae_arima_SBP,rmse_arima_SBP,mape_arima_HR,mae_arima_HR,rmse_arima_HR
0,1127912.00,0.06,8.14,10.01,0.08,6.89,8.30
1,1164638.00,0.03,3.21,4.44,0.02,1.29,1.72
2,1166047.00,0.03,3.61,4.06,0.05,3.33,4.09
3,1193821.00,0.08,8.62,10.18,0.02,1.42,2.11
4,1231804.00,0.16,19.18,22.11,0.06,4.91,5.35
...,...,...,...,...,...,...,...
743,98613334.00,0.10,12.91,13.96,0.02,1.42,1.66
744,98660120.00,0.01,1.60,1.67,0.00,0.26,0.35
745,98716129.00,0.06,9.69,10.94,0.06,3.45,4.73
746,98999464.00,0.02,2.29,2.73,0.02,1.42,2.95


In [9]:
df_vitals_metrics_arima.to_csv("sarima_models_stats.csv", index = False)

In [10]:
df_vitals_metrics_arima[~(df_vitals_metrics_arima['mape_arima_SBP']>1)&~(df_vitals_metrics_arima['mape_arima_HR']>1)].mean()

patient_uuid     48958625.44
mape_arima_SBP          0.05
mae_arima_SBP           6.13
rmse_arima_SBP          7.75
mape_arima_HR           0.05
mae_arima_HR            3.56
rmse_arima_HR           4.58
dtype: float64